# Writing Why: train a model

Train the paper's 16-copy Born language model (or one copy, or the softmax transformer) on TinyStories V2 words, then
let it write and explain its words. Use a GPU. The paper trains on $10^6$ windows (about 90 minutes on a laptop GPU);
the default here is $2\times10^5$. TinyStories (2.2 GB) goes in `data/tinystories/` first: see `data/README.md`.

In [ ]:
#@title Setup (run this first)
# Get the code (in Colab: a fresh copy of the repository; when opened from a clone: that clone)
import os, sys, subprocess
if os.path.exists("writing_why.py"):
    ROOT = ".."
elif os.path.exists("demo/writing_why.py"):
    ROOT = "."
else:
    if not os.path.exists("writing-why/.git"):
        subprocess.run(["git", "clone", "-q", "https://github.com/stein-research/writing-why"], check=True)
    ROOT = "writing-why"
ROOT = os.path.abspath(ROOT)
sys.path.insert(0, os.path.join(ROOT, "demo"))
import writing_why as ww
print("device:", ww.DEV)

In [ ]:
#@title Data
d = os.path.join(ROOT, "data", "tinystories")
need = ["TinyStoriesV2-GPT4-train.txt", "TinyStoriesV2-GPT4-valid.txt"]
missing = [f for f in need if not os.path.exists(os.path.join(d, f))]
print("TinyStories:", "ready" if not missing else f"missing {missing}: see data/README.md")

In [ ]:
#@title Train
MODEL = "born-x8x2:m4,t16,x1,r16,p2"  #@param ["born-x8x2:m4,t16,x1,r16,p2", "born-x8x2:m4,x1,r16,p2", "softmax-w96"]
# born-x8x2:...t16...  16 copies (the paper's main model); without t: one copy; softmax-w96: the softmax transformer
N_TRAIN = int(os.environ.get("WW_N_TRAIN", 200_000))  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}
LR, WARM = ("0.01", "0") if MODEL.startswith("softmax") else ("0.032", "0.02")
TAG = f"my_{MODEL.split(':')[0]}{'_t16' if 't16' in MODEL else ''}_n{N_TRAIN}_s{SEED}"
cmd = [sys.executable, "-u", os.path.join("src", "train.py"), "--model", MODEL, "--task", "tinywords", "--epochs", "1",
       "--lr", LR, "--batch", "64", "--n-train", str(N_TRAIN), "--seed", str(SEED), "--warmup", WARM, "--tag", TAG]
with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for line in p.stdout:
        print(line, end="")
if p.returncode:
    raise RuntimeError("training failed")

## Let it write, and explain a word

A briefly trained model writes poorly, and its query-only term $F_q$ can be large (the current word alone already
moves the held polynomial far): the earlier words' shares then sum to a large negative number, still exactly
$P - 1/V - F_q$. With the paper's $10^6$ windows $F_q$ is about $10^{-7}$.

In [ ]:
mine = ww.load(TAG)
story = ww.write(mine, "Once upon a time", words=120, seed=0)
print(story.text)
if mine.family == "B":
    T = max(t for t, x in enumerate(story.words) if x.isalpha() and t > 4)   # the last word it wrote
    display(ww.show(story, ww.explain(mine, story.ids, T)))